# Vol-surface render mock — validate before touching `charts.tsx`

Prototype the front fix on **real** captured data (no production edit). Two real problems found:

1. **Render** — current `VolSurface` uses `mesh3d` over a point cloud with no colorscale → jagged, default blue. Fix: `type:"surface"` over a gridded Z + green→yellow→red colorscale + a **fixed z-axis** so it stops auto-zooming.
2. **Bug** — `orderedBands` (`charts.tsx`) orders the smile axis by **signed delta**, which is *not* monotonic in strike: a 10Δ put (deep OTM, high IV) lands right next to ATM (low IV) → an artificial **spike** in the middle. Fix: order the axis by **log-moneyness** (strike). This affects the 2D `SmileChart` too.

`plotly.py` matches the `plotly.js` schema, so the validated config ports verbatim.

In [1]:
import math, statistics
from pathlib import Path

import plotly.graph_objects as go
from algotrading.frontend.app import create_app
from algotrading.frontend.context import AppContext
from algotrading.infra.storage import ParquetStore
from fastapi.testclient import TestClient

UNDERLYING, TRADE_DATE = "SX5E", "2026-06-12"
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ctx = AppContext(store_root=REPO / "data", configs_dir=REPO / "configs", store=ParquetStore(REPO / "data"))
with TestClient(create_app(ctx)) as c:
    aj = c.get("/api/analytics", params={"underlying": UNDERLYING, "trade_date": TRADE_DATE}).json()
mats = sorted(aj["maturities"], key=lambda m: m["maturity_years"])
print(f"{UNDERLYING} {TRADE_DATE}:", [m["tenor_label"] for m in mats])

/srv/project/.venv/lib/python3.13/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa
{"logger": "httpx", "level": "INFO", "ts": "2026-06-13T11:24:13.642454Z", "message": "HTTP Request: GET http://testserver/api/analytics?underlying=SX5E&trade_date=2026-06-12 \"HTTP/1.1 200 OK\""}


SX5E 2026-06-12: ['10d', '1m', '3m', '6m', '12m', '18m', '2y', '3y']


In [2]:
# Order bands by MEAN log-moneyness (strike-monotonic) — NOT signed delta. This is the fix for
# the spike. Then build a maturity x band Z-matrix, NaN where a coarse long-dated tenor lacks a
# band (honest holes at the wings).
band_lm = {}
for m in mats:
    for p in m["points"]:
        band_lm.setdefault(p["delta_band"], []).append(p["log_moneyness"])
mean_lm = {b: statistics.mean(v) for b, v in band_lm.items()}
bands = sorted(mean_lm, key=mean_lm.get)
xlm = [mean_lm[b] for b in bands]
ys = [m["maturity_years"] for m in mats]

z = []
for m in mats:
    iv = {p["delta_band"]: p["implied_vol"] for p in m["points"]}
    z.append([iv.get(b, float("nan")) for b in bands])

print("bands (strike order):", bands)
for m, row in zip(mats, z):
    print(f"  {m['tenor_label']:>4}", ["NaN" if isinstance(v, float) and math.isnan(v) else round(v, 3) for v in row])
print("\nEach row should be a smooth monotone skew (puts high -> calls low). No spike.")

bands (strike order): ['10dp', '20dp', '30dp', 'atm', 'atmp', '30dc', '20dc', '10dc']
   10d [0.21, 0.189, 0.176, 0.162, 0.162, 0.154, 0.151, 0.148]
    1m [0.224, 0.196, 0.18, 0.161, 0.161, 0.149, 0.145, 0.142]
    3m [0.248, 0.21, 0.189, 0.167, 0.167, 0.153, 0.147, 0.143]
    6m [0.26, 0.22, 0.198, 0.172, 0.172, 0.156, 0.15, 0.145]
   12m [0.267, 0.226, 0.203, 0.176, 0.176, 0.16, 0.153, 0.147]
   18m ['NaN', 0.225, 0.203, 0.177, 0.177, 0.16, 0.153, 0.147]
    2y ['NaN', 0.23, 0.208, 0.18, 0.18, 0.162, 0.155, 'NaN']
    3y ['NaN', 0.231, 0.209, 0.181, 0.181, 0.163, 0.156, 'NaN']

Each row should be a smooth monotone skew (puts high -> calls low). No spike.


In [3]:
VOL_SCALE = [[0.0, "#1a9850"], [0.25, "#a6d96a"], [0.5, "#fee08b"], [0.75, "#fdae61"], [1.0, "#d73027"]]
Z_RANGE = [0.0, 0.35]  # fixed z-axis (anchored at 0) so the scale is stable across dates


def layout(title):
    return dict(
        title=title,
        scene=dict(
            xaxis=dict(title="log-moneyness"),
            yaxis=dict(title="maturity (y)"),
            zaxis=dict(title="implied vol", range=Z_RANGE),
            aspectratio=dict(x=1.4, y=1.5, z=0.7),
            camera=dict(eye=dict(x=1.8, y=-1.8, z=0.8)),
        ),
        margin=dict(l=0, r=0, t=40, b=0),
        height=560,
    )

## (A) Current — `mesh3d`, delta-ordered axis (jagged blue + spike)

In [4]:
# Reproduce the current look: delta-ordered x, mesh3d point cloud, default colorscale.
dband = {}
for m in mats:
    for p in m["points"]:
        dband.setdefault(p["delta_band"], p["target_delta"])
delta_order = sorted(dband, key=dband.get)
xs, yy, zz = [], [], []
for m in mats:
    iv = {p["delta_band"]: p["implied_vol"] for p in m["points"]}
    for j, b in enumerate(delta_order):
        if b in iv:
            xs.append(j); yy.append(m["maturity_years"]); zz.append(iv[b])
go.Figure(go.Mesh3d(x=xs, y=yy, z=zz, opacity=0.9)).update_layout(**layout("A — mesh3d, delta-ordered (current)"))

## (C) Proposed — `surface`, log-moneyness axis, green→yellow→red, fixed z, gaps bridged

In [5]:
go.Figure(
    go.Surface(z=z, x=xlm, y=ys, colorscale=VOL_SCALE, colorbar=dict(title="IV"), connectgaps=True)
).update_layout(**layout("C — surface, log-moneyness + fixed z (proposed)"))

## Read-out → port to `charts.tsx`

Reordering by log-moneyness turns every smile into a **monotone textbook skew** — the spike was the delta-ordered axis, a real bug, not the data. To port:

1. **`VolSurface`**: `type:"mesh3d"` → `type:"surface"` with this `z`/`x`(log-moneyness)/`y` grid, `colorscale: VOL_SCALE`, `zaxis.range = [0, 0.35]`, `connectgaps`.
2. **`orderedBands` / `smileAxis`**: order by **log-moneyness**, not `target_delta` — fixes the 2D `SmileChart` spike too.
3. **CSS**: explicit height on `.surface-panel .plot` so the 3D scene stops overlapping the greeks panel.
4. The `svi_a` degeneracy flag does not distort the projected IVs (textbook skew) → over-sensitive, a separate lower-priority finding.

## Palette comparison — pick the colormap

The shipped front palette is **pastel** (the dark-theme UI tokens `#a8e6ba`/`#ef9c92`) — too faded, too little high/low contrast, hard to read. Below is the **same** proposed surface; only the `colorscale` changes, so judge readability directly. (Standards: perceptually-uniform colormaps + monotone luminance for readability — [matplotlib/viridis](https://bids.github.io/colormap/); cold=low / hot=high IV convention — [MenthorQ](https://menthorq.com/guide/3d-volatility-surface/).)

- **Plasma** — perceptually uniform (monotone luminance → instantly readable), colorblind-safe, low end is blue (stays visible on a dark background), reads cold→hot like quant tools. **Recommended.**
- **Inferno** — also perceptually uniform, maximum contrast, but its low end is near-black and can vanish into the dark theme.
- **Vivid RdYlGn** — the intuitive finance green→red (vivid, not pastel), but **not** perceptually uniform (the bright central yellow misreads as a peak) and weak for red-green colour blindness.

In [6]:
# Same proposed surface (log-moneyness x, maturity y, fixed z=[0,0.35], gaps bridged) — ONLY the
# colorscale changes. cmin/cmax lock the colour mapping so a given colour means the same IV in all
# three, making the comparison fair. Run this cell: it renders the three candidates in order.
RDYLGN_VIVID = [[0.0, "#1a9850"], [0.25, "#91cf60"], [0.5, "#fee08b"], [0.75, "#fc8d59"], [1.0, "#d73027"]]


def palette_fig(colorscale, title):
    return go.Figure(
        go.Surface(z=z, x=xlm, y=ys, colorscale=colorscale, cmin=0.0, cmax=0.35,
                   colorbar=dict(title="IV"), connectgaps=True)
    ).update_layout(**layout(title))


for cs, title in [
    ("Plasma",      "Plasma — perceptually uniform, cold->hot (RECOMMENDED)"),
    ("Inferno",     "Inferno — max contrast, low end ~ black"),
    (RDYLGN_VIVID,  "Vivid RdYlGn — finance green->red, NOT perceptually uniform"),
]:
    palette_fig(cs, title).show()

## Blue-family comparison

Same surface again — blue colormaps this time. Key trade-off on a **dark theme**: the high-IV end should be the *bright* end so peaks pop and recede into shadow at low IV.

- **Cividis** — perceptually uniform and specifically colorblind-optimised; runs blue→yellow (so not pure blue, but the most readable of the blues).
- **Ice** — perceptually uniform, near-black→blue→white; clean monochrome feel, but the low end can blend into the dark panel.
- **Blues** — the classic ColorBrewer sequential, light→**dark** blue; note its high end is *dark*, so high IV may disappear against the dark background (often wants reversing).
- **YlGnBu** — yellow-green-blue, blue-dominant but multi-hue.
- **Custom navy→cyan→white** — vivid monochrome blue with a *bright* high end (high IV pops on dark), dark navy low end recedes. Tunable.

In [7]:
# Blue-family candidates — same surface, only the colorscale changes (cmin/cmax locked as before).
# A mix of perceptually-uniform maps (Cividis, Ice) and classic sequential blues, plus a vivid
# custom navy->cyan->white tuned for the dark theme (bright high end = high IV pops; dark low end
# recedes). Reuses palette_fig() defined in the cell above — run that cell first.
NAVY_CYAN = [[0.0, "#08306b"], [0.35, "#2171b5"], [0.7, "#41b6e4"], [1.0, "#d9f4ff"]]

for cs, title in [
    ("Cividis", "Cividis - perceptually uniform, colorblind-optimised (blue->yellow)"),
    ("Ice",     "Ice - perceptually uniform, near-black -> blue -> white"),
    ("Blues",   "Blues - classic ColorBrewer, light -> dark blue (high end may blend on dark bg)"),
    ("YlGnBu",  "YlGnBu - yellow-green-blue, blue-dominant multi-hue"),
    (NAVY_CYAN, "Custom navy -> cyan -> white - vivid monochrome blue (bright high end)"),
]:
    palette_fig(cs, title).show()

## Smoothness — prototype A (blueprint-mandated): render from the SVI fit, not the band points

The kinks are not noise — the front plots only **8 strike points per smile** (one per delta band) with linear interpolation, so each smile is a coarse polyline. The blueprint requires the surface to be a **regularized total-variance grid sampled from the fit**, interpolated across maturities in variance space (`05-math-notes:38`, `03-roadmap:191/195`, Eq 22). The SVI parameters are **already persisted** (15 slices for SX5E 2026-06-12), so we can sample the smooth curve `w(k) = a + b(ρ(k−m) + √((k−m)²+σ²))`, `IV = √(w/T)`, on a dense grid — **no re-capture needed**.

Below: the **same** palette (Plasma), coarse band grid (BEFORE) vs the SVI fit sampled on a 41×40 grid (AFTER). The AFTER surface is what approach A serves.

In [8]:
# --- Prototype A (blueprint-mandated): render the nappe from the SMOOTH SVI fit, NOT the 8 band
# points. The blueprint requires a regularized total-variance grid sampled from the fit
# (05-math-notes:38, 03-roadmap:191/195) and interpolation ACROSS maturities in variance space
# (Eq 22). The SVI params are ALREADY persisted (15 slices) -> no re-capture needed to sample them.
import numpy as np
import pyarrow.parquet as pq

sp_path = REPO / "data/derived/surface_parameters/trade_date=2026-06-12/underlying=SX5E/data.parquet"
sp = pq.ParquetFile(str(sp_path)).read().to_pandas().sort_values("maturity_years").reset_index(drop=True)


def svi_w(k, a, b, rho, m, sigma):  # Eq 20, raw-SVI total variance w(k)
    return a + b * (rho * (k - m) + np.sqrt((k - m) ** 2 + sigma ** 2))


K = np.linspace(-0.20, 0.20, 41)  # dense strike axis (vs 8 band pts / 5 persisted grid buckets)
T_fit = sp["maturity_years"].to_numpy()
W_fit = np.vstack([svi_w(K, r.svi_a, r.svi_b, r.svi_rho, r.svi_m, r.svi_sigma) for r in sp.itertuples()])
# Interpolate ACROSS maturities in VARIANCE space (blueprint Eq 22) onto a dense maturity axis.
T = np.linspace(T_fit.min(), T_fit.max(), 40)
W = np.column_stack([np.interp(T, T_fit, W_fit[:, j]) for j in range(K.size)])
IV = np.sqrt(np.maximum(W, 1e-12) / T[:, None])  # IV = sqrt(w / T)

deg = sp[sp["svi_sigma"].abs() < 1e-6]["maturity_years"].round(3).tolist()
print("fitted slices:", len(sp), "| dense grid:", IV.shape, "(maturity x k)")
print("degenerate slices (sigma~0, blueprint: flag + smooth fallback):", deg)

# Before/after on the SAME palette (Plasma): coarse 8-band grid vs smooth SVI sample.
go.Figure(
    go.Surface(z=z, x=xlm, y=ys, colorscale="Plasma", cmin=0.0, cmax=0.35,
               colorbar=dict(title="IV"), connectgaps=True)
).update_layout(**layout("BEFORE - 8 band points, linear interp (current front)")).show()

go.Figure(
    go.Surface(z=IV, x=K, y=T, colorscale="Plasma", cmin=0.0, cmax=0.35, colorbar=dict(title="IV"))
).update_layout(**layout("AFTER (A) - SVI fit sampled 41x40, total-variance interp (SMOOTH)")).show()

fitted slices: 15 | dense grid: (40, 41) (maturity x k)
degenerate slices (sigma~0, blueprint: flag + smooth fallback): [1.764]


## Colour nuance — the real lever is the colour *range*, not the number of stops

The washed-out look isn't "too few colours" (Plasma already blends 256). It's that the colour range is pinned to **[0, 0.35]** while index IV lives in a narrow band (~0.14–0.27 in the body), so the data only ever touches ~40% of the gradient. Two **independent** Plotly levers: `cmin`/`cmax` set the **colour** range (nuance); `zaxis.range` sets the **height** (relief/flatness).

Five variants of the same smooth surface below:

1. **Current** — Plasma, fixed `[0, 0.35]`: washed out and flat (the problem).
2. **Auto-fit** — `cmin/cmax` = data min/max: maximum nuance and relief, but the scale moves every day so colours/heights are **not comparable across dates**.
3. **Fixed tight band `[0.10, 0.30]`** *(recommended)* — full gradient across the index-IV band, real relief, and still a **fixed** scale so dates stay comparable (a vol spike past 30% just saturates the top, like a clipped axis).
4. **Ordered 5-stop blend** on the tight band — distinct-but-perceptually-ordered colours.
5. **Greeks 2D palette blended** on the tight band — your idea, the exact line colours from the dollar-Greeks charts. Note it's not perceptually ordered (green→blue→amber→purple→red), so it reads more "rainbow" than monotone — compare it against 3/4.

In [9]:
# Colour NUANCE — the lever is the colour range (cmin/cmax), not the number of stops. The fixed
# [0,0.35] wastes ~half the scale because index IV sits in a narrow band; tightening cmin/cmax
# spreads the whole colormap across the actual IV range. Reuses the smooth IV/K/T from the cell
# above. zaxis.range is the *height* lever (relief), independent of colour.
iv_lo, iv_hi = float(np.min(IV)), float(np.max(IV))
print(f"actual IV on this surface: {iv_lo:.3f} .. {iv_hi:.3f}  (vs the fixed scale 0.00 .. 0.35)")


def even(colors):  # a plain colour list -> evenly-spaced [pos, colour] stops Plotly accepts
    n = len(colors)
    return [[i / (n - 1), c] for i, c in enumerate(colors)]


GREEKS_BANDS = even(["#a8e6ba", "#8fc7ff", "#f0cf7a", "#d6b3ff", "#ef9c92"])  # the 2D greeks line palette
DISTINCT5 = even(["#2c7fb8", "#41b6c4", "#a1dab4", "#fec44f", "#f03b20"])  # ordered 5-stop, distinct


def surf(colorscale, cmin, cmax, zlo, zhi, title):
    return (
        go.Figure(
            go.Surface(z=IV, x=K, y=T, colorscale=colorscale, cmin=cmin, cmax=cmax, colorbar=dict(title="IV"))
        )
        .update_layout(
            title=title,
            scene=dict(
                xaxis=dict(title="log-moneyness"),
                yaxis=dict(title="maturity (y)"),
                zaxis=dict(title="implied vol", range=[zlo, zhi]),
                aspectratio=dict(x=1.4, y=1.5, z=0.7),
                camera=dict(eye=dict(x=1.8, y=-1.8, z=0.8)),
            ),
            margin=dict(l=0, r=0, t=40, b=0),
            height=560,
        )
    )


surf("Plasma", 0.0, 0.35, 0.0, 0.35, "1) CURRENT - Plasma, fixed [0,0.35] (washed out, flat)").show()
surf("Plasma", iv_lo, iv_hi, iv_lo, iv_hi, "2) Plasma, AUTO-fit to data (max nuance+relief, NOT stable across dates)").show()
surf("Plasma", 0.10, 0.30, 0.10, 0.30, "3) Plasma, fixed TIGHT band [0.10,0.30] (nuance + relief + stable) - RECO").show()
surf(DISTINCT5, 0.10, 0.30, 0.10, 0.30, "4) Ordered 5-stop blend, tight band").show()
surf(GREEKS_BANDS, 0.10, 0.30, 0.10, 0.30, "5) Greeks 2D palette blended, tight band (your idea)").show()

actual IV on this surface: 0.139 .. 0.353  (vs the fixed scale 0.00 .. 0.35)


### Richer spectrum — more stops, same cool→warm order (extending option 4)

Keeping the natural ascending spectrum from option 4 (blue → cyan → green → yellow → red) but adding stops for a smoother, richer blend. `4b`/`4c`/`4d` are hand-made 7/9/12-stop ramps; `4e` is **Turbo**, the canonical smooth full-spectrum map (a perceptually-improved rainbow — many colours, ordered, no hard banding). All on the recommended tight band `[0.10, 0.30]`.

Pick the one that reads best — the more stops, the closer to a continuous spectrum. (Caveat for the record: full-spectrum/rainbow ramps trade a little luminance-monotonicity for vividness; Turbo is the best-engineered of them.)

In [10]:
# Extending option 4: same NATURAL SPECTRUM ORDER (cool -> warm, ascending), more stops for a
# richer blend. Tight band [0.10, 0.30]. Reuses surf()/even() from the cell above. "Turbo" is the
# canonical smooth full-spectrum ordered colormap (Google's perceptually-improved rainbow).
ORIG5 = even(["#2c7fb8", "#41b6c4", "#a1dab4", "#fec44f", "#f03b20"])  # option 4, for reference
SPECTRUM7 = even(["#3a4cc0", "#3f8fce", "#41b6c4", "#7fcdbb", "#addd8e", "#fec44f", "#f03b20"])
SPECTRUM9 = even(
    ["#3a4cc0", "#3f8fce", "#41b6c4", "#7fcdbb", "#a1dab4", "#d9f0a3", "#fee391", "#fe9929", "#f03b20"]
)
SPECTRUM12 = even(
    ["#3022a0", "#3a4cc0", "#3f8fce", "#41b6c4", "#66c2a5", "#a1dab4", "#d9f0a3",
     "#fee391", "#fec44f", "#fe9929", "#f4622e", "#d7191c"]
)

surf(ORIG5, 0.10, 0.30, 0.10, 0.30, "4) original 5-stop (reference)").show()
surf(SPECTRUM7, 0.10, 0.30, 0.10, 0.30, "4b) 7-stop spectrum").show()
surf(SPECTRUM9, 0.10, 0.30, 0.10, 0.30, "4c) 9-stop spectrum").show()
surf(SPECTRUM12, 0.10, 0.30, 0.10, 0.30, "4d) 12-stop spectrum (finest hand-made)").show()
surf("Turbo", 0.10, 0.30, 0.10, 0.30, "4e) Turbo (named full-spectrum, smoothest)").show()